# പാഠം 05 - ഏജൻറിക് RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ഏജന്റിക് RAG എന്താണ്?

പരമ്പരാഗത RAG ഒരു നിശ്ചിത പൈപ്പ്ലൈൻ പിന്തുടരുന്നു: ഡോക്യമെന്റുകൾ തിരയുക, പിന്നെ ഒരു പ്രതികരണം സൃഷ്‌ടിക്കുക. **ഏജന്റിക് RAG** അതിനപ്പുറം പോകുന്നു, ഏജന്റിന് **എപ്പോഴാണ്** എങ്ങനെ വിവരങ്ങൾ തിരയാമെന്ന് സ്വയം തീരുമാനിക്കാൻ സ്വാതന്ത്ര്യം നൽകുന്നു.

ഏജന്റിക് RAG ഉപയോഗിച്ച്, ഏജന്റ് ചെയ്യുന്നത്:
- ഒരു ചോദ്യം മറുപടി നൽകുന്നതിനു മുമ്പ് തിരയൽ ആവശ്യമുണ്ടോ എന്ന് **നിർണ്ണയിക്കുക**
- ഏത് ഡാറ്റ സ്രോതസ്സോ ടൂളോ ചോദിക്കാമെന്ന് **തിരഞ്ഞെടുക്കുക**
- നേടിയ ഫലങ്ങൾ **അവലോകനം ചെയ്ത്** ആദ്യ ശ്രമംക്ഷമമല്ലെങ്കിൽ തുടർ തിരയലുകൾ നടത്തുക
- നിരവധി തിരയൽ ഘട്ടങ്ങളിൽ നിന്നുള്ള വിവരങ്ങൾ **ഒന്നിച്ച്** സമഗ്രമായ മറുപടി ഉണ്ടാക്കുക

ഇതിലൂടെ ഏജന്റ് ഒരു സ്ഥിരമായ തിരയൽ-തിരക്കൽ പൈപ്പ്ലൈനിനേക്കാൾ കൂടുതൽ ലവചനപരവും കൃത്യവുമാകുന്നു.


## ഒരു സെർച്ച് ഉപകരണം സൃഷ്ടിക്കൽ

ഏജന്റിക് RAG-യിൽ, ബെഹിര്ഗത ഡാറ്റ സ്രോതസുകൾ **ഉപകരണങ്ങളായി** പൊതിയപ്പെട്ടിട്ടുണ്ട്, ഏജന്റ് ആവശ്യപ്പെടുമ്പോൾ ഇത് വിനിയോഗിക്കാനാകുന്നു. ഇതിലൂടെ, ഏജന്റ് തിരയലിനെ ഒരു നിർബന്ധമായ ഘട്ടമല്ല, മറിച്ച് ഓർമ്മപ്പെടുത്താനുള്ള മറ്റൊരു പ്രവർത്തനമായാണ് കാണിക്കുന്നത്.

താഴെ, ഒരു യാത്രാ അറിവ് താനും നിർവ്വചിച്ച് ഏജൻറ് അത് വിളിച്ച് ലക്ഷ്യസ്ഥല വിവരങ്ങൾ തേടാൻ സാധിക്കുന്ന ഉപകരണമായി വെളിപ്പെടുത്തുന്നു.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ഏജന്റ് നിർമ്മിക്കൽ

ഇപ്പോൾ ഞങ്ങൾ ഒരു ഏജന്റ് സൃഷ്ടിക്കുന്നു, അത് **ഉത്തരം നൽകുന്നതിന് മുമ്പായി എല്ലായ്പോഴും വിവരങ്ങൾ തിരിച്ചെടുക്കാൻ** നിർദ്ദേശിച്ചിരിക്കുന്നു. ഏജന്റ് തന്റെ പ്രതികരണങ്ങൾക്ക് സ്വന്തം പരിശീലന ഡാറ്റയിൽ ആശ്രയിക്കുന്നതിനുപകരം, അറിവ് അടിസ്ഥാനത്തിൽ ഉറപ്പാക്കാൻ `search_travel_knowledge` ഉപകരണം ഉപയോഗിക്കുന്നു.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ആവർത്തിച്ചു ലഭിക്കൽ — മെക്കർ-ചെക്കർ മാതൃക

ഏജന്റിക് RAG ന്റെ ഒരു പ്രധാന ഗുണം **ആവർത്തിച്ചുള്ള ലഭ്യത** ആണ്. ഏജന്റ് തൻറെ ആദിമ കണ്ടെത്തലുകൾ സ്ഥിരീകരിക്കാൻ, സുതാര്യമാക്കാൻ, അല്ലെങ്കിൽ വിപുലമാക്കാൻ നിരവധി തവണ തിരയൽ നടത്താൻ കഴിയും — "മെക്കർ-ചെക്കർ" പ്രവൃത്തി രീതിയോട് സമാനമായി:

1. **മെക്കർപടി**: ഏജന്റ് ആരംഭിക വിവരങ്ങൾ തിരഞ്ഞെടുത്ത് ഒരു മറുപടി രൂപീകരിക്കുന്നു.
2. **ചെക്കർപടി**: ഏജന്റ് വിശദാംശങ്ങൾ സ്ഥിരീകരിക്കാൻ അല്ലെങ്കിൽ ശൂന്യമായിടങ്ങൾ നിറയ്ക്കാൻ അധിക തിരയലുകൾ അനുഷ്ഠിക്കുന്നു.

താഴെ, ഏgent ന് ബെഹിതമായ ലക്ഷ്യസ്ഥലങ്ങൾ താരതമ്യം ചെയ്യേണ്ട ഒരു ചോദ്യം ചോദിക്കുന്നു, ഇത് അത് പല തവണ തിരയാൻ പ്രേരിപ്പിക്കുന്നു.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## സംഗ്രഹം

ഈ പാഠത്തില്‍ Microsoft Agent Framework ഉപയോഗിച്ച് **Agentic RAG** സിസ്റ്റം എങ്ങിനെ നിർമ്മിക്കാമെന്ന് നിങ്ങള്‍ പഠിച്ചു:

- **Agentic RAG** ഏജന്റുകൾ സ്വതന്ത്രമായി വിവരങ്ങൾ എപ്പോഴറിയണമെന്ന് തീരുമാനിക്കാൻ അനുവദിക്കുന്നു, എങ്ങനെ സുദൃഢമാക്കുന്നതിന് പകരം രേഖപ്പെടുത്തിയ പദവി ഡൈനാമിക് ആവും.
- **ഡാറ്റാ സ്രോതസ്സുകളായി ഉപകരണങ്ങൾ**: Azure AI Search പോലുള്ള ബാഹ്യ നോളജ് ബേസുകൾ ഉപകരണങ്ങളായി ഏജന്റ് ഉപയോഗിക്കാൻ പൊതിഞ്ഞിരിക്കുന്നു.
- **പുനരാവൃത്തി റിട്ടീവ്**: മെക്കർ-ചെക്കർ മാതൃക ഏജന്റിന് പല റിട്ടീവൽ റൗണ്ടുകൾ നടത്താൻ അനുവദിക്കുന്നു — തിരയൽ, സ്ഥിരീകരണം, നവീകരണം — ഒടുവിൽ ഒരു അന്തിമ ഉത്തരവാദി പുറപ്പെടുവിക്കുന്നതിന് മുമ്പായുള്ള പ്രക്രിയ.

ഉത്പാദനത്തിലുള്ള, വലിയ തോതിലുള്ള യാത്രാ ഡოკ്യുമെന്റ് റീറ്റ്രീവലിന് വേണ്ടി, മെമ്മറിയിലുള്ള `TRAVEL_KNOWLEDGE_BASE` യെ യഥാർത്ഥ Azure AI Search ഇൻഡക്സിൽ മാറ്റും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
